# Custosell - Offline & Sync Architecture

**Version:** 1.0  
**Last updated:** June 2026  
**Scope:** Frontend (Electron + React) ↔ Laravel API ↔ MySQL  
**Canonical doc:** [architecture.md](./architecture.md)  
**Related:** [sales.md](./sales.md), [expenses.md](./expenses.md), [auth.md](./auth.md)

This notebook documents the **client/server data strategy** used in Custosell: when to use IndexedDB, when to use the API, how sync works, and the fixes applied for shift linking, registration, and UI banners.

---
## 1. Executive summary

| Layer | Role | When used |
|-------|------|-----------|
| **MySQL (server)** | System of record | Online reads & writes; final truth after sync |
| **IndexedDB (`CustosellOffline`)** | Offline cache + mutation outbox | Completely offline; local sessions; draining pending sync |
| **React Query cache** | Fast in-memory UI layer | Online (and fallback when API unreachable) |
| **localStorage** | Auth session mirror | Online auth hydrate (avoids opening IndexedDB unnecessarily) |

**Rule of thumb:**

- **Online (normal server session)** → read/write the **server**; IndexedDB stays closed unless there is **pending sync work**.
- **Offline** → read/write the **client** (IndexedDB + cache); mutations are **queued** for later.
- **Online + pending queue** → server reads with **merge overlay** of local pending rows; coordinator **drains** the queue to the server.

This is the standard **offline-first + outbox pattern** used by POS and field apps - not a full client replica of the server database.

---
## 2. Architecture diagram

```mermaid
flowchart TB
  subgraph ui [UI Layer]
    Pages[Pages / Components]
    RQ[React Query cache]
  end

  subgraph strategy [Read strategy]
    ROS[readWithOfflineStrategy]
  end

  subgraph online [Online / slow]
    API[Laravel API]
    MySQL[(MySQL)]
  end

  subgraph client [Client persistence]
    IDB[(IndexedDB CustosellOffline)]
    MQ[mutation queue]
    LS[localSales / localShifts / ...]
  end

  subgraph sync [Sync coordinator]
    SC[syncCoordinator]
    AUTH[syncAuthEngine]
    ENG[syncEngine]
  end

  Pages --> ROS
  ROS -->|offline| IDB
  ROS -->|online| API
  API --> MySQL
  ROS --> RQ

  Pages -->|write offline| IDB
  IDB --> MQ
  IDB --> LS

  SC --> AUTH
  SC --> ENG
  MQ --> SC
  SC -->|replay| API
```

---
## 3. Connectivity & “completely offline”

Network state lives in Redux (`networkSlice`). A probe hits the API (and optionally external endpoints) to classify status:

| `systemStatus` | Meaning | Treated as offline? |
|----------------|---------|---------------------|
| `online` | API reachable, latency normal | No |
| `slow` | API reachable, high latency | **No** (still server-first) |
| `offline` | API unreachable | **Yes** |

### Detectors (must stay aligned)

| Function / selector | Logic |
|---------------------|-------|
| `selectIsCompletelyOffline` | `systemStatus === 'offline'` |
| `isCompletelyOffline()` | Same - Redux probe only |
| `isOfflineMode()` | `getSystemStatus() === 'offline'` |
| `shouldUseClientStorage()` | `isOfflineMode()` |
| `shouldFetchFromServer()` | `!isOfflineMode()` |

**Important:** `slow` is **not** offline. Users on slow connections still use the server; we do not switch to IndexedDB for reads.

**Files:** `networkSlice.ts`, `connectivityCheck.ts`, `offlineQueryUtils.ts`, `useNetworkStatusMonitor.ts`

---
## 4. Read strategy

All major list/detail queries use `readWithOfflineStrategy`:

```typescript
// offlineReadStrategy.ts (conceptual)
if (shouldUseClientStorage()) {
  return readFromClient();           // IndexedDB + React Query cache
}
try {
  return await fetchFromServer();    // API → often merged with pending local rows
} catch (err) {
  if (networkFailure || 401-without-token || 404) {
    return readFromClient();         // graceful fallback
  }
  throw err;
}
```

### Online read with pending work

Many `fetchFromServer` implementations **merge** server data with pending IndexedDB rows (e.g. sales list, shift sales overlay). That is intentional: the user may have created records offline that are not on the server yet.

### Dashboard vs My Shift

| Screen | Scopes by | Overlay module |
|--------|-----------|----------------|
| **Dashboard** | Calendar date | `offlineSalesSummary.ts` |
| **My Shift** | `shift_id` | `offlineShiftOverlay.ts` |

My Shift uses **server baselines** (`shiftKeys.serverSales`, `serverExpenses`) plus overlay on every read - same pattern as Dashboard.

---
## 5. Write strategy

| Condition | Behaviour |
|-----------|----------|
| `shouldCompleteMutationLocally()` (= completely offline) | Complete instantly in UI; persist to IndexedDB; enqueue mutation |
| Online, API succeeds | POST/PUT/DELETE to server; update React Query cache |
| Online, network failure (no HTTP response) | Fall back to local completion + queue (same as offline path for that action) |

**Mutations are not batched for all entity types.** Sales use `POST /sales/batch` during sync; expenses use multipart and sync individually.

**Files:** `completeOfflineSale.ts`, `completeOfflineExpense.ts`, `completeOfflineShift.ts`, `mutationQueue.ts`

---
## 6. IndexedDB - when it opens

Database name: **`CustosellOffline`** (version 11).

`canUseOfflineDbNow()` returns true when **any** of:

1. `systemStatus === 'offline'`
2. `auth.isLocalSession` or `auth.pendingAuthSync`
3. `localStorage` hint `custosell_pending_queue === '1'`

Otherwise `getOfflineDb()` rejects with `OfflineDbUnavailableError` (internal - never shown verbatim to users).

### Store inventory (high level)

| Store | Purpose |
|-------|--------|
| `mutations` | Outbound API queue (outbox) |
| `localSales` | Pending sales |
| `localShifts` | Pending shift open/close |
| `localExpenses` | Pending expenses |
| `localProducts`, `localCategories`, … | Other pending entities |
| `localAuth` | Offline registration / device login |
| `stock` / `adjustments` | Stock ledger overrides |
| `secureSecrets` / `secureKeys` | Encrypted auth (offline path) |

### Auth storage when online

- **`loadAuthSession`:** `localStorage` + Electron secure store first; IndexedDB only if `canUseOfflineDbNow()`.
- **`saveAuthSession`:** Always writes `localStorage`; IndexedDB write only when `canUseOfflineDbNow()`.

**Files:** `offlineDb.ts`, `offlineQueryUtils.ts`, `secureStorage.ts`, `mutationQueue.ts`

---
## 7. Sync coordinator

Triggered by `useOfflineSync` on:

- App bootstrap (if pending work or `pendingAuthSync`)
- Reconnect after `offline` → `online`/`slow`

### Tier order

```mermaid
flowchart LR
  T0[Tier 0: Auth] --> T1[Tier 1: Shifts open]
  T1 --> T2[Tier 2: Sales batch]
  T2 --> T3[Tier 3: Expenses]
  T3 --> T4[Tier 4: Shifts close]
  T4 --> T5[Products / categories / staff / ...]
```

### Auth sync (tier 0)

- Register: `POST /businesses/register` then `POST /auth/login`
- **Idempotent register:** duplicate email → skip register, login only (`isDuplicateEmailError`)
- **Online register split:** if register succeeds but login times out → queue **login only** (`completePendingLoginAfterRegister`)
- Preserves shift context across login (`capturePreservedShiftContext`)

### Shift ID remap

Offline clock-in uses **negative local shift ids**. Sync maps them to server ids before sales/expenses with that `shift_id` are sent. Unmapped negative ids **defer** sync (`isUnresolvedLocalShiftId`).

**Files:** `syncCoordinator.ts`, `syncAuthEngine.ts`, `syncEngine.ts`, `syncSalesBatch.ts`, `offlineShiftOverlay.ts`

---
## 8. Shift linking (sales & expenses)

### Problem we fixed

Sales were saved with `shift_id: null` while an active shift existed → visible on **Dashboard** (date filter) but not **My Shift** (`shift_id` filter).

### Expense pattern (reference)

```typescript
// ExpenseForm.tsx
const activeShiftId = shiftId ?? authShiftId;
if (!isEditing && activeShiftId) formData.append('shift_id', String(activeShiftId));
```

My Shift passes `shiftId={shift?.id ?? authUser?.shift_id}` into the form.

### Sales - same pattern

| Layer | Implementation |
|-------|----------------|
| Resolver | `resolveActiveShiftId(shiftId?)` → `shiftId ?? auth.shift_id` |
| New Sale UI | `useActiveShift()` + resolver before checkout |
| Offline/online create | `withSaleShiftLink(payload)` |
| Backend fallback | `SaleService::resolveShiftId()` - active shift if client omits `shift_id` |
| Batch sync | `SaleController::batch` validates `sales.*.shift_id` |

### Defence in depth

1. **UI** resolves shift from active shift query + auth
2. **Client** applies `withSaleShiftLink` before API/queue
3. **Server** assigns active shift when `shift_id` is null

---
## 9. UI banners & error surfacing

| Banner | When shown | When hidden |
|--------|------------|-------------|
| **OfflineBanner** | `systemStatus === 'offline'` (dismissible) | Online or slow |
| **AuthPendingBanner** | `local_` token + `pendingAuthSync` | Server token; or while sync progress active |
| **SyncProgressBanner** | Sync running / paused / failed / partial success | Idle or dismissed |

### Session normalization

On bootstrap, `normalizeStoredAuthSession` clears stale flags for **server tokens**:

- `pendingAuthSync: false`
- `isLocalSession: false`

Prevents “connect to the internet” banner when already online with a real account.

### Internal errors

IndexedDB gate and connection errors are sanitized via `sanitizeErrorMessage` / `sanitizeSyncErrorMessage` - users see *“Sync failed. Try again in a moment.”* not internal gate text.

**Files:** `AuthPendingBanner.tsx`, `OfflineBanner.tsx`, `SyncProgressBanner.tsx`, `normalizeAuthSession.ts`, `AuthBootstrap.tsx`

---
## 10. Registration duplicate-email flow

```mermaid
sequenceDiagram
  participant User
  participant App
  participant IDB as IndexedDB queue
  participant API

  User->>App: Register offline
  App->>IDB: enqueue POST /businesses/register
  Note over App: local_ token, pendingAuthSync=true

  User->>App: Back online
  App->>API: POST /businesses/register
  API-->>App: 422 duplicate email (or 500 before fix)
  App->>API: POST /auth/login (skip register)
  API-->>App: user + server token
  App->>App: remap local business/user ids
  App->>IDB: remove auth mutation
```

Backend: `BusinessRegisterRequest` validates `unique:users,email` → 422 instead of SQL error.

---
## 11. Decision matrix (quick reference)

| User state | Reads | Writes | IndexedDB | Banners |
|------------|-------|--------|-----------|--------|
| Online, server session, no pending queue | API + RQ cache | API | Closed | None |
| Online, server session, pending sales | API + merge local | API + drain queue | Open until empty | Sync progress |
| Online, `local_` session, pending register | Client + sync | Queue → API | Open | Auth pending or sync |
| Completely offline | Client only | Local + queue | Open | Offline + auth pending |
| Slow connection | API (same as online) | API (timeout → local fallback) | Only if pending hint | None unless sync running |

---
## 12. Key file index

### Frontend - core strategy

| File | Responsibility |
|------|----------------|
| `offlineReadStrategy.ts` | Server-first reads, client fallback |
| `offlineQueryUtils.ts` | Offline flags, queue hint, error sanitization |
| `offlineDb.ts` | IndexedDB singleton + gate |
| `mutationQueue.ts` | Outbox queue |
| `syncCoordinator.ts` | Ordered sync loop |
| `syncPendingIfOnline.ts` | Entry point when connectivity returns |
| `useOfflineSync.ts` | React hook - bootstrap + reconnect sync |

### Frontend - domain

| File | Responsibility |
|------|----------------|
| `offlineShiftOverlay.ts` | My Shift merge + shift id remap |
| `resolveActiveShiftId.ts` | Shift linking helper |
| `completeOfflineSale.ts` | Local sale + `withSaleShiftLink` |
| `syncAuthEngine.ts` | Auth register/login sync |
| `syncSalesBatch.ts` | Sales batch upload |
| `normalizeAuthSession.ts` | Clear stale offline flags |
| `secureStorage.ts` | Auth persist; localStorage-first online |

### Backend

| File | Responsibility |
|------|----------------|
| `SaleService.php` | `resolveShiftId()` fallback on create |
| `SaleController.php` | Batch validation includes `shift_id` |
| `BusinessRegisterRequest.php` | Unique email validation on register |

---
## 13. Is this the correct / industry-standard approach?

**Yes.** Custosell implements a well-established pattern:

1. **Server-authoritative when online** - MySQL is the source of truth.
2. **Offline-first writes** - user actions complete locally without blocking on the network.
3. **Outbox pattern** - `mutations` store replays work when connectivity returns.
4. **Merge-on-read** - pending local rows overlay server data until sync completes.
5. **Minimal IndexedDB surface online** - avoids connection churn, “database closing” errors, and surprise banners.

Alternatives (not used here):

- **Full client replica (PouchDB/CouchDB sync)** - heavier; overkill for this POS scope.
- **Online-only with no queue** - simpler but unusable when connectivity drops.
- **IndexedDB as primary even when online** - causes stale data and the bugs we fixed.

Custosell’s split matches what teams building offline-capable React/Electron POS apps typically ship.

---
## 14. Operational notes

### Clearing a stuck local session

1. Log out
2. Sign in **online** with email/password (gets server Sanctum token)
3. Optional: DevTools → Application → IndexedDB → delete `CustosellOffline` if corruption suspected

### Verifying shift linkage in DB

```sql
SELECT id, receipt_number, shift_id, user_id, sale_date FROM sales ORDER BY id DESC LIMIT 10;
SELECT id, user_id, status, clock_in, clock_out FROM shifts WHERE status = 'active';
```

### Local API (dev)

- Frontend: `http://localhost:5173`
- Backend: `http://localhost:8000/api/v1`
- MySQL: `127.0.0.1:3307`, database `custosell`